# Understanding Hash
Session 05 (2026-10-17): how hashing works and how a hash table uses it.

## 1. What a hash is
A hash function maps any input to a fixed-size integer. Same input, same output.

In [ ]:
def toy_hash(s):
    return sum(ord(c) for c in s)

print(toy_hash("cat"), toy_hash("cat"), toy_hash("act"))  # 'act' collides with 'cat'

## 2. Python's `hash()`

In [ ]:
print(hash(42), hash(3.0), hash((1, 2)))
print(hash("hello"))  # changes between interpreter runs (hash randomization)

In [ ]:
try:
    hash([1, 2])
except TypeError as e:
    print("lists are unhashable:", e)

## 3. How a hash table works
Hash the key, take it modulo the number of buckets to get an index, store the value there.

In [ ]:
buckets = 8
for key in ["apple", "banana", "cherry", "date"]:
    print(key, "->", hash(key) % buckets)

## 4. Collisions and resizing
Two keys can land in the same bucket. This table uses chaining (a list per bucket) and doubles in size when it gets too full.

In [ ]:
class HashTable:
    def __init__(self, size=8):
        self.buckets = [[] for _ in range(size)]
        self.count = 0

    def _bucket(self, key):
        return self.buckets[hash(key) % len(self.buckets)]

    def set(self, key, value):
        bucket = self._bucket(key)
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))
        self.count += 1
        if self.count / len(self.buckets) > 0.75:
            self._resize()

    def get(self, key):
        for k, v in self._bucket(key):
            if k == key:
                return v
        raise KeyError(key)

    def delete(self, key):
        bucket = self._bucket(key)
        for i, (k, _) in enumerate(bucket):
            if k == key:
                del bucket[i]
                self.count -= 1
                return
        raise KeyError(key)

    def _resize(self):
        old = [item for b in self.buckets for item in b]
        self.buckets = [[] for _ in range(len(self.buckets) * 2)]
        self.count = 0
        for k, v in old:
            self.set(k, v)

In [ ]:
t = HashTable()
for i, w in enumerate(["a", "b", "c", "d", "e", "f", "g", "h"]):
    t.set(w, i)
print(len(t.buckets), t.get("c"))
t.delete("c")
print(t.count)

## 5. Why lookups are fast
Average O(1) for `dict`, versus O(n) scanning a list.

In [ ]:
import timeit
n = 100_000
lst, dct = list(range(n)), {i: i for i in range(n)}
print("list:", timeit.timeit(lambda: (n - 1) in lst, number=100))
print("dict:", timeit.timeit(lambda: (n - 1) in dct, number=100))